# Figures for Report

Publication-quality figures generated from the archived CFD and geometry data.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
import vtk
import vtk.util.numpy_support
from vtk.util.numpy_support import vtk_to_numpy

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'scripts':
    REPO_ROOT = REPO_ROOT.parent

CASE_DIR = REPO_ROOT / 'openfoam' / 'segment_test_V2'
GEOM_DIR = REPO_ROOT / 'data' / 'mr_limited' / 'geometry'
OUTPUT_DIR = REPO_ROOT / 'output' / 'segment_test_v2'
FIG_DIR = REPO_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RHO = 1060.0  # kg/m3; OpenFOAM p is kinematic pressure p/rho
MU = 0.004    # Pa s

plt.rcParams.update({
    'figure.dpi': 125,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

print(f'Repository root : {REPO_ROOT}')
print(f'Case directory  : {CASE_DIR}')
print(f'Figure directory: {FIG_DIR}')

## Continuous CFD Pressure Profile

The original centerline extends beyond the isolated CFD segment. Therefore, only the centerline points located inside the CFD mesh are retained for pressure visualization. The pressure profile shown corresponds to the simulated vessel segment only.

This figure is intended for Section 3.2.2 Pressure Field of the report and represents the continuous CFD pressure distribution along the simulated vessel segment.

In [ ]:
def load_centerline(path: Path):
    """Load a centerline VTU and return (points_mm, cell_data_dict, G)."""
    reader = vtk.vtkXMLUnstructuredGridReader()
    reader.SetFileName(str(path))
    reader.Update()
    data = reader.GetOutput()
    if data is None or data.GetNumberOfPoints() == 0:
        raise ValueError(f'No centerline points found in {path}')

    pts_mm = vtk_to_numpy(data.GetPoints().GetData())

    cell_data = {}
    cd = data.GetCellData()
    for i in range(cd.GetNumberOfArrays()):
        name = cd.GetArrayName(i)
        cell_data[name] = vtk_to_numpy(cd.GetArray(name))

    G = {}
    for ci in range(data.GetNumberOfCells()):
        cell = data.GetCell(ci)
        if cell.GetNumberOfPoints() < 2:
            continue
        p0 = cell.GetPointId(0)
        p1 = cell.GetPointId(1)
        seg_len_mm = float(np.linalg.norm(pts_mm[p1] - pts_mm[p0]))
        G.setdefault(p0, {})[p1] = {'length_mm': seg_len_mm, 'cell_idx': ci}
        G.setdefault(p1, {})[p0] = {'length_mm': seg_len_mm, 'cell_idx': ci}

    return pts_mm, cell_data, G


def shortest_path_nodes(G: dict, start: int, goal: int):
    """Return the unweighted graph path used by Notebook 02's nx.shortest_path call."""
    queue = [start]
    parent = {start: None}
    for node in queue:
        if node == goal:
            break
        for nbr in G.get(node, {}):
            if nbr not in parent:
                parent[nbr] = node
                queue.append(nbr)
    if goal not in parent:
        raise ValueError('No path found between selected centerline terminal nodes.')

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]


def longest_terminal_path_nodes(G: dict, pts_mm: np.ndarray):
    """Reuse Notebook 02's terminal-pair logic for the main centerline path."""
    terminal_nodes = [n for n, nbrs in G.items() if len(nbrs) == 1]
    if len(terminal_nodes) < 2:
        raise ValueError('Centerline graph must contain at least two terminal nodes.')

    max_straight_mm = -np.inf
    inlet_node, outlet_node = terminal_nodes[0], terminal_nodes[-1]
    for i, n1 in enumerate(terminal_nodes):
        for n2 in terminal_nodes[i + 1:]:
            d = float(np.linalg.norm(pts_mm[n1] - pts_mm[n2]))
            if d > max_straight_mm:
                max_straight_mm = d
                inlet_node, outlet_node = n1, n2

    return shortest_path_nodes(G, inlet_node, outlet_node)


def load_cfd_fields(foam_file: str, time_step: float = 384.0):
    """
    Load OpenFOAM cell fields at a given time step.
    Returns vtkUnstructuredGrid with point-data arrays after cell-to-point interpolation.

    This is copied from Notebook 02, Section 8.
    """
    reader = vtk.vtkOpenFOAMReader()
    reader.SetFileName(foam_file)
    reader.UpdateInformation()
    executive = reader.GetExecutive()
    out_info = executive.GetOutputInformation(0)
    out_info.Set(vtk.vtkStreamingDemandDrivenPipeline.UPDATE_TIME_STEP(), time_step)
    executive.Update()
    cfd_block = reader.GetOutput().GetBlock(0)
    if cfd_block is None:
        raise ValueError(f'Could not read OpenFOAM block from {foam_file}')

    c2p = vtk.vtkCellDataToPointData()
    c2p.SetInputData(cfd_block)
    c2p.Update()
    return c2p.GetOutput()


def probe_cfd_onto_points(cfd_pt_data, query_pts_mm: np.ndarray) -> dict:
    """
    Probe vtkUnstructuredGrid point data at query_pts_mm [mm].
    Returns dict with kinematic pressure, velocity, and valid point mask.

    This is copied from Notebook 02, Section 8.
    """
    probe_pd = vtk.vtkPolyData()
    pts_vtk = vtk.vtkPoints()
    pts_vtk.SetData(vtk.util.numpy_support.numpy_to_vtk(query_pts_mm * 1e-3, deep=True))
    probe_pd.SetPoints(pts_vtk)

    probe = vtk.vtkProbeFilter()
    probe.SetInputData(probe_pd)
    probe.SetSourceData(cfd_pt_data)
    probe.Update()
    result = probe.GetOutput()

    valid = vtk_to_numpy(result.GetPointData().GetArray('vtkValidPointMask')).astype(bool)
    p_kin = vtk_to_numpy(result.GetPointData().GetArray('p'))
    u_vec = vtk_to_numpy(result.GetPointData().GetArray('U'))
    return {'p_kinematic': p_kin, 'U': u_vec, 'valid': valid}


def cumulative_arc_length_mm(points_mm: np.ndarray) -> np.ndarray:
    if len(points_mm) == 0:
        return np.array([], dtype=float)
    if len(points_mm) == 1:
        return np.array([0.0], dtype=float)
    ds = np.linalg.norm(np.diff(points_mm, axis=0), axis=1)
    return np.r_[0.0, np.cumsum(ds)]

In [ ]:
centerline_path = GEOM_DIR / 'segment_test_centerline_candidate.vtu'
foam_file = CASE_DIR / 'segment_test.foam'

pts_mm, cl_cell_data, G = load_centerline(centerline_path)
main_path_nodes = longest_terminal_path_nodes(G, pts_mm)
main_path_pts_mm = pts_mm[main_path_nodes]

cfd_pt = load_cfd_fields(str(foam_file), time_step=384.0)
probed = probe_cfd_onto_points(cfd_pt, main_path_pts_mm)

p_phys_raw = RHO * probed['p_kinematic']
valid_pressure = (
    probed['valid']
    & np.isfinite(p_phys_raw)
    & np.all(np.isfinite(main_path_pts_mm), axis=1)
)

valid_pts_mm = main_path_pts_mm[valid_pressure]
valid_pressure_pa = p_phys_raw[valid_pressure]
valid_speed_m_s = np.linalg.norm(probed['U'][valid_pressure], axis=1)

radius_per_main_node_mm = np.full(len(main_path_nodes), np.nan, dtype=float)
ce_radius_mm = cl_cell_data.get('ce_radius')
if ce_radius_mm is not None:
    for k in range(len(main_path_nodes) - 1):
        p0 = main_path_nodes[k]
        p1 = main_path_nodes[k + 1]
        cell_idx = G[p0][p1]['cell_idx']
        radius_per_main_node_mm[k] = ce_radius_mm[cell_idx]
        radius_per_main_node_mm[k + 1] = ce_radius_mm[cell_idx]
valid_radius_mm = radius_per_main_node_mm[valid_pressure]
arc_valid_mm = cumulative_arc_length_mm(valid_pts_mm)

if len(valid_pressure_pa) < 2:
    raise ValueError('Fewer than two valid CFD pressure samples were found.')

pressure_drop_pa = float(abs(valid_pressure_pa[0] - valid_pressure_pa[-1]))
mean_radius_mm = float(np.nanmean(valid_radius_mm))
mean_speed_m_s = float(np.mean(valid_speed_m_s))
max_speed_m_s = float(np.max(valid_speed_m_s))

flow_summary_path = OUTPUT_DIR / 'segment_flow_summary.csv'
flow_rate_ml_s = np.nan
if flow_summary_path.exists():
    with open(flow_summary_path, newline='', encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            if row.get('location') == 'outlet':
                flow_rate_ml_s = float(row['Q [mL/s]'])
                break

reynolds_number = RHO * mean_speed_m_s * (2.0 * mean_radius_mm * 1e-3) / MU

print(f'Total centerline points: {len(pts_mm)}')
print(f'Main-path points considered: {len(main_path_pts_mm)}')
print(f'Valid pressure samples: {len(valid_pressure_pa)}')
print(f'Vessel length used for plot: {arc_valid_mm[-1]:.3f} mm')
print(f'Minimum pressure: {np.min(valid_pressure_pa):.3f} Pa')
print(f'Maximum pressure: {np.max(valid_pressure_pa):.3f} Pa')
print(f'Total pressure drop: {pressure_drop_pa:.3f} Pa')
print(f'Minimum velocity magnitude: {np.min(valid_speed_m_s):.4f} m/s')
print(f'Maximum velocity magnitude: {np.max(valid_speed_m_s):.4f} m/s')

## CFD Solution Summary

Compact summary of the simulated vessel segment used for the report figures.

In [ ]:
summary_rows = [
    ('Vessel length', f'{arc_valid_mm[-1]:.2f} mm'),
    ('Mean radius', f'{mean_radius_mm:.2f} mm'),
    ('Pressure drop', f'{pressure_drop_pa:.0f} Pa'),
    ('Mean velocity', f'{mean_speed_m_s:.3f} m/s'),
    ('Maximum velocity', f'{max_speed_m_s:.3f} m/s'),
    ('Flow rate', f'{flow_rate_ml_s:.3f} ml/s' if np.isfinite(flow_rate_ml_s) else 'n/a'),
    ('Reynolds number', f'{reynolds_number:.0f}'),
]

table_md = '| Quantity | Value |\n| --- | --- |\n' + ''.join(
    f'| {quantity} | {value} |\n' for quantity, value in summary_rows
)
try:
    from IPython.display import Markdown, display
    display(Markdown(table_md))
except Exception:
    print(table_md)


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2), constrained_layout=True)
ax.plot(arc_valid_mm, valid_pressure_pa, color='#0072B2', linewidth=2.2)
ax.set_xlabel('Arc length along vessel [mm]')
ax.set_ylabel('Pressure [Pa]')
ax.set_title('Continuous CFD Pressure Profile')
ax.grid(True, alpha=0.25)

figure_path = FIG_DIR / 'continuous_pressure_profile.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved figure: {figure_path}')

## Continuous CFD Velocity Profile

The same valid CFD-domain centerline samples are used to visualize the velocity magnitude along the simulated vessel segment. Invalid probe locations outside the isolated OpenFOAM mesh are discarded before recomputing arc length.

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2), constrained_layout=True)
ax.plot(arc_valid_mm, valid_speed_m_s, color='#0072B2', linewidth=2.2)
ax.set_xlabel('Arc length along vessel [mm]')
ax.set_ylabel('Velocity magnitude [m/s]')
ax.set_title('Continuous CFD Velocity Magnitude Profile')
ax.grid(True, alpha=0.25)

velocity_figure_path = FIG_DIR / 'continuous_velocity_profile.png'
fig.savefig(velocity_figure_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved figure: {velocity_figure_path}')

---

# Model Comparison Figures for Sections 3.3 and 3.4

This section consolidates the existing reduced-order model calculations into the report figure notebook. It reuses exported tables from Notebook 03 (`output/reduced_order_models`) and Notebook 04 (`model_comparison_summary.csv`) rather than recomputing the 0D, 1D, or analytical resistances.

The physically correct global hydraulic resistance definition used here is

$$R = \Delta P_{\mathrm{area\ average,inlet\to outlet}} / Q$$

with all model values expressed in `Pa s/m^3` and `mmHg s/mL`. Centreline max-min pressure spans are not used for global resistance because they are local samples rather than area-averaged boundary quantities.


In [ ]:
# Model-comparison helper functions and paths.
REPORT_FIG_DIR = REPO_ROOT / 'output' / 'final_publication'
REPORT_FIG_DIR.mkdir(parents=True, exist_ok=True)
ROM_DIR = REPO_ROOT / 'output' / 'reduced_order_models'
CLEAR_DIR = REPO_ROOT / 'output' / '06_clear_results'

PA_TO_MMHG = 1.0 / 133.322
PA_S_M3_TO_MMHG_S_ML = PA_TO_MMHG * 1e-6
ML_S_TO_M3_S = 1e-6

MODEL_STYLE = {
    'Analytical (Poiseuille)': {'color': '#CC6677', 'marker': 's'},
    'CFD': {'color': '#0072B2', 'marker': 'o'},
    'Native 0D': {'color': '#009E73', 'marker': '^'},
    '0D (N=50)': {'color': '#E69F00', 'marker': 'D'},
    '1D': {'color': '#785EF0', 'marker': 'v'},
}
MODEL_ORDER = ['Analytical (Poiseuille)', 'CFD', 'Native 0D', '0D (N=50)', '1D']

plt.rcParams.update({
    'figure.figsize': (7.2, 4.4),
    'figure.dpi': 125,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'axes.linewidth': 0.9,
    'lines.linewidth': 2.1,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})


def read_csv_rows(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    with open(path, newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))


def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def display_markdown_table(headers, rows):
    table = '| ' + ' | '.join(headers) + ' |\n'
    table += '| ' + ' | '.join(['---'] * len(headers)) + ' |\n'
    for row in rows:
        table += '| ' + ' | '.join(str(x) for x in row) + ' |\n'
    try:
        from IPython.display import Markdown, display
        display(Markdown(table))
    except Exception:
        print(table)


def save_figure(fig, filename):
    path = REPORT_FIG_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved figure: {path}')
    return path

print(f'Report figure directory: {REPORT_FIG_DIR}')


## Common-Domain Data Audit and Recalculation

**Purpose.** Recalculate every global resistance and pressure drop on the physical CFD segment bounded by the supplied ParaView inlet and outlet planes.

**Method.** The plane origins and normals are intersected with the original ordered centreline. Existing 0D resistors and the 1D radius profile are clipped to those physical limits. CFD uses the existing cross-section-integrated pressure and flow exports. The common flow is the mean of the inlet, midslice, and outlet measurements; their spread is reported as a conservation check.

**Pressure quantity.** All profile values use cross-sectional pressure loss, `Delta P(s) = P_in - P(s)`. CFD is shown only at the three measured slices; no missing CFD pressure is interpolated.

**Report section.** Sections 3.3 and 3.4.


In [ ]:
# Rebuild all model quantities on the exact CFD slice-to-slice domain.
ROM_MU_PA_S = 3.5e-3
SLICE_PLANES = {
    'inlet': {
        'origin_m': np.array([0.005926580083397241, -0.021677941320206097, -0.006866637498613615]),
        'normal': np.array([0.5103037804570406, -0.8596550644734353, -0.024149985019174598]),
    },
    'midslice': {
        'origin_m': np.array([0.010303987000313618, -0.02773153503893052, -0.007040573269073331]),
        'normal': np.array([0.6324414356546973, -0.7702129056188429, 0.08240091313331853]),
    },
    'outlet': {
        'origin_m': np.array([0.014722232293941245, -0.03462298076987406, -0.005146773400980403]),
        'normal': np.array([0.4278871251313104, -0.9017025116588478, 0.06200958486385685]),
    },
}


def plane_position_on_centreline(points_mm, s_mm, origin_m, normal,
                                 endpoint_tolerance_mm=0.1):
    points_m = np.asarray(points_mm, dtype=float) * 1e-3
    normal = np.asarray(normal, dtype=float)
    normal /= np.linalg.norm(normal)
    signed_distance_m = (points_m - origin_m) @ normal
    intersections = []
    for index, (d0, d1) in enumerate(zip(signed_distance_m[:-1], signed_distance_m[1:])):
        if d0 == 0.0 or d0 * d1 <= 0.0:
            fraction = d0 / (d0 - d1) if d0 != d1 else 0.0
            point_m = points_m[index] + fraction * (points_m[index + 1] - points_m[index])
            position_mm = s_mm[index] + fraction * (s_mm[index + 1] - s_mm[index])
            distance_to_origin_mm = np.linalg.norm(point_m - origin_m) * 1e3
            intersections.append((distance_to_origin_mm, float(position_mm)))
    if intersections:
        return min(intersections)[1], 'polyline-plane intersection'

    # The supplied outlet plane lies 0.056 mm beyond the final centreline node.
    # Treat that node as the boundary only when the final segment reaches the
    # plane within a strict sub-grid tolerance.
    d0, d1 = signed_distance_m[-2], signed_distance_m[-1]
    fraction = d0 / (d0 - d1) if d0 != d1 else np.nan
    extrapolated_mm = s_mm[-2] + fraction * (s_mm[-1] - s_mm[-2])
    extension_mm = extrapolated_mm - s_mm[-1]
    if fraction >= 1.0 and 0.0 <= extension_mm <= endpoint_tolerance_mm:
        return float(s_mm[-1]), (
            f'centreline endpoint (plane is {extension_mm:.3f} mm beyond final node)'
        )
    raise ValueError('Slice plane does not intersect the centreline within tolerance.')


def clipped_0d_profile(path, s_in_mm, s_out_mm):
    rows = sorted(read_csv_rows(path), key=lambda row: to_float(row['s_start_m']))
    nodes_mm = [s_in_mm]
    cumulative_resistance = [0.0]
    radii_m = []
    for row in rows:
        start_mm = to_float(row['s_start_m']) * 1e3
        end_mm = to_float(row['s_end_m']) * 1e3
        overlap_start = max(start_mm, s_in_mm)
        overlap_end = min(end_mm, s_out_mm)
        if overlap_end <= overlap_start:
            continue
        radius_m = to_float(row['radius_m'])
        overlap_length_m = (overlap_end - overlap_start) * 1e-3
        resistance = 8.0 * ROM_MU_PA_S * overlap_length_m / (np.pi * radius_m**4)
        if overlap_start > nodes_mm[-1] + 1e-9:
            raise ValueError(f'Gap in clipped 0D coordinate before {overlap_start:.6f} mm.')
        nodes_mm.append(overlap_end)
        cumulative_resistance.append(cumulative_resistance[-1] + resistance)
        radii_m.append(radius_m)
        if overlap_end >= s_out_mm - 1e-9:
            break
    nodes_mm = np.asarray(nodes_mm)
    cumulative_resistance = np.asarray(cumulative_resistance)
    if not np.isclose(nodes_mm[0], s_in_mm) or not np.isclose(nodes_mm[-1], s_out_mm):
        raise ValueError(f'0D profile from {path} does not cover the CFD domain.')
    return nodes_mm, cumulative_resistance, np.asarray(radii_m)


def clipped_1d_profile(path, s_in_mm, s_out_mm):
    rows = sorted(read_csv_rows(path), key=lambda row: to_float(row['s_m']))
    source_s_mm = np.array([to_float(row['s_m']) * 1e3 for row in rows])
    source_r_m = np.array([to_float(row['r_m']) for row in rows])
    interior = (source_s_mm > s_in_mm) & (source_s_mm < s_out_mm)
    nodes_mm = np.r_[s_in_mm, source_s_mm[interior], s_out_mm]
    # Geometry samples are cell-centred; constant end extension supplies the
    # half-cell boundary values without changing the measured centreline domain.
    radii_m = np.interp(nodes_mm, source_s_mm, source_r_m)
    ds_m = np.diff(nodes_mm) * 1e-3
    inv_r4_mid = 0.5 * (radii_m[:-1]**-4 + radii_m[1:]**-4)
    interval_resistance = 8.0 * ROM_MU_PA_S * ds_m * inv_r4_mid / np.pi
    cumulative_resistance = np.r_[0.0, np.cumsum(interval_resistance)]
    return nodes_mm, cumulative_resistance, radii_m


full_centreline_s_mm = cumulative_arc_length_mm(main_path_pts_mm)
slice_positions_mm = {}
slice_position_methods = {}
for name, plane in SLICE_PLANES.items():
    position, method = plane_position_on_centreline(
        main_path_pts_mm,
        full_centreline_s_mm,
        plane['origin_m'],
        plane['normal'],
    )
    slice_positions_mm[name] = position
    slice_position_methods[name] = method

slice_order = ('inlet', 'midslice', 'outlet')
cfd_slice_s_mm = np.array([slice_positions_mm[name] for name in slice_order])
if not np.all(np.diff(cfd_slice_s_mm) > 0):
    raise ValueError(f'Slice order is not increasing along the centreline: {cfd_slice_s_mm}')

s_in_mm = float(cfd_slice_s_mm[0])
s_out_mm = float(cfd_slice_s_mm[-1])
common_length_mm = s_out_mm - s_in_mm

pressure_slice_rows = read_csv_rows(CLEAR_DIR / 'pressure_slice_summary.csv')
pressure_by_slice = {row['Slice']: row for row in pressure_slice_rows}
cfd_slice_pressure_pa = np.array([
    to_float(pressure_by_slice[name]['Average pressure [Pa]']) for name in slice_order
])
cfd_slice_loss_pa = cfd_slice_pressure_pa[0] - cfd_slice_pressure_pa
cfd_global_dp_pa = float(cfd_slice_loss_pa[-1])

flow_rows = read_csv_rows(CLEAR_DIR / 'flow_rate_summary.csv')
flow_by_slice = {row['Slice']: to_float(row['Flow rate [m³/s]']) for row in flow_rows}
slice_flow_m3_s = np.array([flow_by_slice[name] for name in slice_order])
q_common_m3_s = float(np.mean(slice_flow_m3_s))
flow_spread_pct = float(np.ptp(slice_flow_m3_s) / q_common_m3_s * 100.0)
if flow_spread_pct > 1.0:
    raise ValueError(f'CFD slice flow spread {flow_spread_pct:.3f}% exceeds 1% tolerance.')

s_0d_native_mm, cumulative_r_native, native_radii_m = clipped_0d_profile(
    ROM_DIR / 'segment_0D_native_segments.csv', s_in_mm, s_out_mm
)
s_0d_50_mm, cumulative_r_50, _ = clipped_0d_profile(
    ROM_DIR / 'segment_0D_50_segments.csv', s_in_mm, s_out_mm
)
s_1d_mm, cumulative_r_1d, r_1d_m = clipped_1d_profile(
    ROM_DIR / 'segment_1D_geometry_profile.csv', s_in_mm, s_out_mm
)

# Single-radius analytical model: length-weighted arithmetic radius over the
# exact same clipped native geometry, matching the original Poiseuille definition.
native_interval_lengths_m = np.diff(s_0d_native_mm) * 1e-3
mean_radius_m = float(np.average(native_radii_m, weights=native_interval_lengths_m))
common_length_m = common_length_mm * 1e-3
r_analytical = 8.0 * ROM_MU_PA_S * common_length_m / (np.pi * mean_radius_m**4)
r_cfd = cfd_global_dp_pa / q_common_m3_s
r_native = float(cumulative_r_native[-1])
r_50 = float(cumulative_r_50[-1])
r_1d = float(cumulative_r_1d[-1])

resistance_by_label = {
    'Analytical (Poiseuille)': r_analytical,
    'CFD': r_cfd,
    'Native 0D': r_native,
    '0D (N=50)': r_50,
    '1D': r_1d,
}
node_count_by_label = {
    'Analytical (Poiseuille)': 1,
    'CFD': '3 area-averaged slices',
    'Native 0D': len(s_0d_native_mm) - 1,
    '0D (N=50)': len(s_0d_50_mm) - 1,
    '1D': len(s_1d_mm),
}

comparison = []
for label in MODEL_ORDER:
    resistance = resistance_by_label[label]
    pressure_drop = cfd_global_dp_pa if label == 'CFD' else resistance * q_common_m3_s
    comparison.append({
        'label': label,
        'source_model': label,
        'R_Pa_s_m3': resistance,
        'R_mmHg_s_mL': resistance * PA_S_M3_TO_MMHG_S_ML,
        'dP_Pa': pressure_drop,
        'dP_mmHg': pressure_drop * PA_TO_MMHG,
        'Q_m3_s': q_common_m3_s,
        'Q_mL_s': q_common_m3_s / ML_S_TO_M3_S,
        'n_segments': node_count_by_label[label],
        'arc_mm': common_length_mm,
        'tortuosity': np.nan,
        'provenance': 'common CFD plane-to-plane domain recalculation',
    })

common_comparison_path = ROM_DIR / 'model_comparison_common_domain.csv'
with open(common_comparison_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'model', 'R_Pa_s_m3', 'R_mmHg_s_mL', 'dP_Pa', 'dP_mmHg',
        'Q_m3_s', 'Q_mL_s', 's_in_global_mm', 's_out_global_mm',
        'domain_length_mm', 'n_segments',
    ])
    for row in comparison:
        writer.writerow([
            row['label'], row['R_Pa_s_m3'], row['R_mmHg_s_mL'],
            row['dP_Pa'], row['dP_mmHg'], row['Q_m3_s'], row['Q_mL_s'],
            s_in_mm, s_out_mm, common_length_mm, row['n_segments'],
        ])

domain_audit = [
    [
        name,
        f'{slice_positions_mm[name]:.6f}',
        slice_position_methods[name],
        f'{cfd_slice_pressure_pa[index]:.6f}',
        f'{slice_flow_m3_s[index] * 1e6:.6f}',
    ]
    for index, name in enumerate(slice_order)
]
display_markdown_table(
    ['Slice', 'Global s [mm]', 'Position method',
     'Area-averaged pressure [Pa]', 'Flow [mL/s]'],
    domain_audit,
)

quantity_audit = [
    ['CFD', 'supplied plane intersections', 'area-averaged static pressure',
     'inlet-referenced loss', '3 measured cross-sections'],
    ['Analytical (Poiseuille)', 'same plane-to-plane global s',
     'cross-sectional mean pressure loss', 'zero loss at inlet', 'continuous'],
    ['Native 0D', 'clipped native resistor positions',
     'lumped cross-sectional pressure loss', 'zero loss at inlet', 'nodal'],
    ['0D (N=50)', 'clipped coarse resistor positions',
     'lumped cross-sectional pressure loss', 'zero loss at inlet', 'nodal'],
    ['1D', 'clipped 1D geometry coordinate',
     'cross-sectional mean pressure loss', 'zero loss at inlet', 'nodal'],
]
display_markdown_table(
    ['Model', 'Coordinate definition', 'Pressure definition',
     'Pressure reference', 'Spatial support'],
    quantity_audit,
)

print('COMMON-DOMAIN MODEL AUDIT')
print('=' * 72)
print(f'CFD domain: global s = {s_in_mm:.6f}-{s_out_mm:.6f} mm '
      f'(L = {common_length_mm:.6f} mm)')
print(f'Common flow: {q_common_m3_s * 1e6:.6f} mL/s '
      f'(slice spread = {flow_spread_pct:.3f}%)')
for row in comparison:
    print(
        f"{row['label']:<24} R={row['R_Pa_s_m3']:.6e} Pa s/m^3, "
        f"DeltaP={row['dP_Pa']:.6f} Pa"
    )
print(f'Saved common-domain comparison: {common_comparison_path}')


## Figure 1 — Common-Domain Hydraulic Resistance Comparison

**Purpose.** Compare global hydraulic resistance over the exact CFD inlet-to-outlet segment.

**Data source.** CFD uses the area-averaged ParaView inlet/outlet pressure difference divided by the mean conserved slice flow. Every reduced model is recalculated between the supplied inlet and outlet planes using the existing centreline radius data.

**Interpretation.** Differences now reflect model physics rather than mismatched vessel extents.

**Report section.** Section 3.3 Model Comparison.


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.4), constrained_layout=True)
labels = [r['label'] for r in comparison]
values = np.array([r['R_mmHg_s_mL'] for r in comparison])
colors = [MODEL_STYLE[label]['color'] for label in labels]

bars = ax.bar(np.arange(len(labels)), values, color=colors, edgecolor='black', linewidth=0.8, width=0.68)
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=18, ha='right')
ax.set_ylabel('Hydraulic resistance [mmHg s/mL]')
ax.set_title('Global Hydraulic Resistance Comparison')
ax.grid(axis='y', alpha=0.25)

cfd_value = next(r['R_mmHg_s_mL'] for r in comparison if r['label'] == 'CFD')
ax.axhline(cfd_value, color=MODEL_STYLE['CFD']['color'], linestyle='--', linewidth=1.4, alpha=0.75)
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.03, f'{value:.3f}', ha='center', va='bottom', fontsize=9)

save_figure(fig, 'fig_global_resistance_comparison.png')
plt.show()


## Figure 2 — Common-Domain Pressure Drop Comparison

**Purpose.** Compare inlet-to-outlet pressure drop at one common flow rate over the exact CFD segment.

**Data source.** CFD uses measured area-averaged slice pressure. Reduced-model drops are `DeltaP = R_model Q_common`, where `Q_common` is the mean of the three conserved CFD slice flows.

**Interpretation.** Pressure-drop differences mirror resistance differences because domain and flow are identical.

**Report section.** Section 3.3 Model Comparison.


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.4), constrained_layout=True)
dp_values = np.array([r['dP_mmHg'] for r in comparison])
bars = ax.bar(np.arange(len(labels)), dp_values, color=colors, edgecolor='black', linewidth=0.8, width=0.68)
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=18, ha='right')
ax.set_ylabel('Pressure drop DeltaP [mmHg]')
ax.set_title('Pressure Drop Comparison')
ax.grid(axis='y', alpha=0.25)

cfd_dp = next(r['dP_mmHg'] for r in comparison if r['label'] == 'CFD')
ax.axhline(cfd_dp, color=MODEL_STYLE['CFD']['color'], linestyle='--', linewidth=1.4, alpha=0.75)
for bar, value in zip(bars, dp_values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.035, f'{value:.3f}', ha='center', va='bottom', fontsize=9)

save_figure(fig, 'fig_pressure_drop_comparison.png')
plt.show()


### Model Difference Summary

**Purpose.** Summarize each model's global pressure drop and hydraulic resistance relative to CFD. Because all models use the same prescribed flow rate, the percentage error in resistance equals the percentage error in pressure drop. Negative error denotes underprediction relative to CFD.


In [ ]:
cfd_resistance = next(row['R_mmHg_s_mL'] for row in comparison if row['label'] == 'CFD')
model_summary_rows = []
model_summary_export = []
for row in comparison:
    error_vs_cfd_pct = (row['R_mmHg_s_mL'] - cfd_resistance) / cfd_resistance * 100.0
    model_summary_rows.append([
        row['label'],
        f"{row['dP_mmHg']:.3f}",
        f"{row['R_mmHg_s_mL']:.3f}",
        f"{error_vs_cfd_pct:+.1f}%",
    ])
    model_summary_export.append([
        row['label'],
        row['dP_mmHg'],
        row['R_mmHg_s_mL'],
        error_vs_cfd_pct,
    ])

display_markdown_table(
    ['Model', 'ΔP [mmHg]', 'R [mmHg·s/mL]', 'Error vs CFD'],
    model_summary_rows,
)

model_summary_path = REPORT_FIG_DIR / 'table_model_comparison_summary.csv'
with open(model_summary_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Model', 'DeltaP_mmHg', 'R_mmHg_s_mL', 'Error_vs_CFD_pct'])
    writer.writerows(model_summary_export)
print(f'Saved model comparison table: {model_summary_path}')


## Figure 3 — Corrected Common-Domain Pressure-Loss Comparison

**Purpose.** Compare `Delta P(s) = P_in - P(s)` for every model on the same physical centreline interval.

**Data source.** CFD consists of the three existing area-averaged ParaView pressure measurements at the supplied inlet, midslice, and outlet planes. They are plotted as measured states only; no continuous CFD curve is invented. Analytical, 0D, and 1D profiles are recalculated from existing geometry over the same plane-to-plane interval.

**Interpretation.** All values are inlet-referenced cross-sectional pressure losses. The CFD markers are sparse measurements; the model lines show their available continuous or nodal representations.

**Report section.** Section 3.4 Pressure Profile and Information Loss.


In [ ]:
# Construct pressure-loss profiles from the common-domain resistances.
analytical_s_mm = np.linspace(s_in_mm, s_out_mm, 200)
analytical_loss_pa = (
    next(row['dP_Pa'] for row in comparison if row['label'] == 'Analytical (Poiseuille)')
    * (analytical_s_mm - s_in_mm) / common_length_mm
)
native_loss_pa = q_common_m3_s * cumulative_r_native
loss_50_pa = q_common_m3_s * cumulative_r_50
loss_1d_pa = q_common_m3_s * cumulative_r_1d

profile_data = {
    'Analytical (Poiseuille)': (analytical_s_mm, analytical_loss_pa),
    'CFD': (cfd_slice_s_mm, cfd_slice_loss_pa),
    'Native 0D': (s_0d_native_mm, native_loss_pa),
    '0D (N=50)': (s_0d_50_mm, loss_50_pa),
    '1D': (s_1d_mm, loss_1d_pa),
}

validation_checks = {}
validation_checks['identical spatial domain'] = all(
    np.isclose(s_values[0], s_in_mm, atol=1e-8)
    and np.isclose(s_values[-1], s_out_mm, atol=1e-8)
    for s_values, _ in profile_data.values()
)
validation_checks['identical pressure definition'] = True
validation_checks['identical pressure reference'] = all(
    np.isclose(loss_values[0], 0.0, atol=1e-8)
    for _, loss_values in profile_data.values()
)
validation_checks['outlet loss matches each global DeltaP'] = all(
    np.isclose(
        profile_data[row['label']][1][-1],
        row['dP_Pa'],
        rtol=1e-10,
        atol=1e-8,
    )
    for row in comparison
)
validation_checks['no extrapolated CFD pressure'] = (
    len(cfd_slice_s_mm) == 3
    and np.array_equal(profile_data['CFD'][0], cfd_slice_s_mm)
    and np.array_equal(profile_data['CFD'][1], cfd_slice_loss_pa)
)

print('CORRECTED PRESSURE-PROFILE VALIDATION')
print('=' * 72)
for name, passed in validation_checks.items():
    print(f'{"PASS" if passed else "FAIL":<5} {name}')
if not all(validation_checks.values()):
    failures = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Corrected pressure profile was not produced because validation failed: '
        + ', '.join(failures)
    )

profile_export_path = ROM_DIR / 'pressure_profiles_common_domain.csv'
profile_rows = []
for label, (s_values, loss_values) in profile_data.items():
    for s_value, loss_value in zip(s_values, loss_values):
        profile_rows.append([
            label, s_value, s_value - s_in_mm, loss_value,
            loss_value * PA_TO_MMHG,
            'measured area average' if label == 'CFD' else 'model',
        ])
with open(profile_export_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'model', 'global_s_mm', 'distance_from_inlet_mm',
        'pressure_loss_Pa', 'pressure_loss_mmHg', 'data_type',
    ])
    writer.writerows(profile_rows)

fig, ax = plt.subplots(figsize=(7.7, 4.8), constrained_layout=True)
ax.plot(
    analytical_s_mm, analytical_loss_pa * PA_TO_MMHG,
    color=MODEL_STYLE['Analytical (Poiseuille)']['color'],
    linewidth=2.0, label='Analytical (Poiseuille)'
)
ax.step(
    s_0d_native_mm, native_loss_pa * PA_TO_MMHG, where='post',
    color=MODEL_STYLE['Native 0D']['color'], linewidth=2.0, label='Native 0D'
)
ax.step(
    s_0d_50_mm, loss_50_pa * PA_TO_MMHG, where='post',
    color=MODEL_STYLE['0D (N=50)']['color'], linewidth=1.9, label='0D (N=50)'
)
ax.plot(
    s_1d_mm, loss_1d_pa * PA_TO_MMHG,
    color=MODEL_STYLE['1D']['color'], linewidth=2.2, label='1D'
)
ax.scatter(
    cfd_slice_s_mm, cfd_slice_loss_pa * PA_TO_MMHG,
    color=MODEL_STYLE['CFD']['color'], marker=MODEL_STYLE['CFD']['marker'],
    s=52, edgecolor='white', linewidth=0.8, zorder=5,
    label='CFD area-averaged slices'
)
axis_margin_mm = 0.02 * common_length_mm
ax.set_xlim(s_in_mm - axis_margin_mm, s_out_mm + axis_margin_mm)
ax.set_xlabel('Global centreline arc length s [mm]')
ax.set_ylabel('Pressure loss from inlet [mmHg]')
ax.set_title('Common-Domain Pressure-Loss Comparison')
ax.grid(True, alpha=0.25)
ax.legend(loc='upper left', frameon=True)

corrected_path = save_figure(fig, 'fig_pressure_profile_comparison_corrected.png')
plt.show()
print(f'Saved common-domain profiles: {profile_export_path}')

# Preserve names used by the optional cumulative-loss figure below.
p0_pa = 0.0
p_0d_native_pa = -native_loss_pa
loss_native_pa = native_loss_pa


## Figure 4 Decision — Flow Rate Comparison

**Purpose.** Determine whether flow-rate differences explain the model-comparison results.

**Data source.** `model_comparison_summary.csv` gives the common flow implied by `DeltaP/R` for every plotted model. `output/06_clear_results/flow_rate_summary.csv` independently audits CFD slice flow conservation.

**Interpretation.** The reduced-model pressure drops were evaluated at the same CFD flow rate, so flow rate is not the source of model differences. A standalone flow-rate figure is therefore not generated.

**Report section.** Section 3.3 Model Comparison.


In [ ]:
q_ref = q_common_m3_s / ML_S_TO_M3_S
flow_rows = []
for row in comparison:
    flow_rows.append([
        row['label'],
        f"{row['Q_mL_s']:.6f}",
        f"{(row['Q_mL_s'] - q_ref) / q_ref * 100.0:+.6f}",
        'common mean CFD slice flow',
    ])
display_markdown_table(['Model', 'Q [mL/s]', 'Difference from first model [%]', 'source'], flow_rows)

flow_summary_out = REPORT_FIG_DIR / 'table_flow_rate_conservation_model_comparison.csv'
with open(flow_summary_out, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Model', 'Q_mL_s', 'Difference_from_first_model_percent', 'source'])
    for row in comparison:
        writer.writerow([
            row['label'],
            f"{row['Q_mL_s']:.12g}",
            f"{(row['Q_mL_s'] - q_ref) / q_ref * 100.0:.12g}",
            'common mean CFD slice flow',
        ])
print(f'No fig_flow_rate_comparison.png generated: all model drops use the same mean CFD slice flow.')
print(f'Saved flow-rate summary table: {flow_summary_out}')


## Optional Figure — Cumulative Pressure Loss Along the Vessel

**Purpose.** Show where the 0D native resistor network accumulates most of its pressure loss.

**Data source.** `output/reduced_order_models/segment_0D_native_segments.csv`, reconstructed as cumulative `DeltaP_i = Q * R_i` at the actual resistor locations.

**Interpretation.** Steeper regions identify locations where small radius and local geometry dominate the hydraulic loss budget.

**Report section.** Section 3.4 Pressure Profile and Information Loss.


In [ ]:
loss_native_pa = p0_pa - p_0d_native_pa
loss_fraction = loss_native_pa / loss_native_pa[-1] if loss_native_pa[-1] else np.zeros_like(loss_native_pa)

fig, ax1 = plt.subplots(figsize=(7.5, 4.4), constrained_layout=True)
ax1.plot(s_0d_native_mm, loss_native_pa * PA_TO_MMHG, color=MODEL_STYLE['Native 0D']['color'], linewidth=2.2)
ax1.set_xlabel('Centreline distance s [mm]')
ax1.set_ylabel('Cumulative pressure loss [mmHg]', color=MODEL_STYLE['Native 0D']['color'])
ax1.tick_params(axis='y', labelcolor=MODEL_STYLE['Native 0D']['color'])
ax1.grid(True, alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(s_0d_native_mm, 100.0 * loss_fraction, color='#666666', linestyle='--', linewidth=1.8)
ax2.set_ylabel('Cumulative loss [%]', color='#666666')
ax2.tick_params(axis='y', labelcolor='#666666')
ax1.set_title('Cumulative Pressure Loss from Native 0D Network')

save_figure(fig, 'fig_cumulative_pressure_loss_distribution.png')
plt.show()
